# CRISP-DM — Risque de panne par équipement

Ce notebook est limité à la classification du risque de panne. 

## 1. Business Understanding

Prédire si le nombre de curatifs du mois suivant dépassera la médiane historique de chaque équipement, afin de prioriser la maintenance.

In [1]:
# 2. Data Understanding et 3. Data Preparation
# La fonction build_dataset de retrain_model conserve toutes les variables utiles :
# lags 1-3, fenêtres glissantes 3 mois, MTBF, disponibilité, arrêts,
# ratio curatif/préventif, volumes et durées.
from retrain_model import get_engine, build_dataset

engine = get_engine()
interventions, training, scoring, features = build_dataset(engine)
print(f'Equipements : {interventions.equip_id.nunique()}')
print(f'Observations : {len(training)} | Features : {len(features)}')
training[['equip_id', 'annee_mois', 'curatif_mois_suivant']].head()

Equipements : 950
Observations : 6414 | Features : 16


,equip_id,annee_mois,curatif_mois_suivant
1,9,2025-07-01,0.0
2,9,2025-08-01,0.0
3,9,2025-09-01,0.0
4,9,2025-10-01,0.0
5,9,2025-11-01,0.0


In [2]:
# 4. Modeling et 5. Evaluation
# Split temporel 80/20 ; comparaison des modèles ; production = Random Forest 
training = training.sort_values(['annee_mois', 'equip_id']).reset_index(drop=True)
split = int(len(training) * 0.8)
print(f'Train temporel : {split} | Test temporel : {len(training) - split}')
print('Cible : curatifs M+1 > médiane historique de l équipement')

Train temporel : 5131 | Test temporel : 1283
Cible : curatifs M+1 > médiane historique de l équipement


In [3]:
# 6. Deployment
# Lance le pipeline complet : champion/challenger (tolérance AUC 0.005),
# prédictions calibrées et catégories par percentiles 66 et 85.
from retrain_model import main
main()

# Sorties : best_model.pkl, predictions_risque.csv et model_meta.json.

RÉENTRAÎNEMENT — RISQUE DE PANNE PAR ÉQUIPEMENT
CRISP-DM : Business Understanding > Data Understanding > Data Preparation
           Modeling > Evaluation > Deployment
Cible : curatifs du mois suivant > médiane historique de l'équipement

[1/4] Extraction et préparation des données...
  Équipements : 950 | interventions : 8,836
  Observations entraînement : 6,414 | features : 16
  Classes : risque cible=1 : 1572 (24.5 %) | cible=0 : 4842
  Fingerprint données : 86f9143d6f686304 | Période : 2025-02 → 2026-03

[2/4] Modeling — split temporel : train=5,131, test=1,283
  Benchmark des modèles en cours...
    Logistic Regression: AUC=0.7108 | Accuracy=0.5425 | F1=0.6084 | Precision=0.4423 | Recall=0.9744
      Matrice de confusion : VP=456 | FP=575 | FN=12 | VN=240
    Random Forest: AUC=0.6801 | Accuracy=0.5783 | F1=0.5562 | Precision=0.4514 | Recall=0.7244
      Matrice de confusion : VP=339 | FP=412 | FN=129 | VN=403
    Gradient Boosting + Calibration: AUC=0.7426 | Accuracy=0.6500 | F1=